In [ ]:
"""
I-BEAM BAYESIAN OPTIMIZATION WITH STABILITY RATIO
5D Search Space: (b, r, dH, B, R) where R = M_cr/M_yield
Heteroscedastic noise: higher near the lateral-torsional buckling boundary.
Plots produced for b, r, dH, R (B is a GP input but not plotted).
"""

import pandas as pd
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.stats import norm
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# MASTER CONTROLS
# ==========================================
EXPLORE_EXPLOIT_DIAL = 0.05
GENERATE_MIXED = True
GENERATE_UCB = True
GENERATE_EI = True

# ==========================================
# PHYSICS CONSTANTS
# ==========================================
TOTAL_HEIGHT = 25.0
B_FIXED = 16.0
MIN_WEB_THICKNESS = 0.8
MIN_FLANGE_WIDTH = 8.0
MAX_WEB_RATIO = 2/3
MATERIAL_DENSITY = 1240
LENGTH_M = 0.2023
YIELD_STRENGTH = 76000000
E_MODULUS = 2.5e9
G_MODULUS = E_MODULUS / 2.6
C1_3PT = 1.35

BOUNDS_5D = {
    'b': (0.8, 10.67),
    'r': (0.0, 4.0),
    'delta_H': (-6.0, 4.0),
    'B': (8.0, 16.0),
    'R': (0.0, 6.0),
}
PARAM_KEYS = list(BOUNDS_5D.keys())
PLOT_DIMS = [0, 1, 2, 4]
PLOT_NAMES = ['b_web (mm)', 'r_fillet (mm)', 'dH (mm)', 'R (stability)']

COL_MAP = {'H': 'H_web_height', 'B': 'B_flange_width', 'b': 'b_web_thick',
           'r': 'r_fillet', 'target': 'Str/w (N/g)'}

def get_acq_params(dial):
    xi_mod = 0.0001 + dial * 0.0199
    kappa_mod = 0.2 + dial * 1.8
    return {
        'EI': {'moderate': xi_mod, 'exploit': 0.0001, 'explore': 0.02},
        'UCB': {'moderate': kappa_mod, 'exploit': 0.2, 'explore': 2.0}
    }

ACQ_CONFIG = get_acq_params(EXPLORE_EXPLOIT_DIAL)

# ==========================================
# PHYSICS CALCULATIONS
# ==========================================
def calc_I(H, h, B, b):
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return (H_m**3 * b_m)/12 + 2*((h_m**3 * B_m)/12 + h_m*B_m*((H_m+h_m)/2)**2)

def calc_Iy(H, h, B, b):
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return (H_m * b_m**3)/12 + 2*(h_m * B_m**3)/12

def calc_J(H, h, B, b):
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return (H_m * b_m**3 + 2*B_m * h_m**3) / 3

def calc_mass(H, h, B, b):
    H_m, h_m, B_m, b_m = H/1000, h/1000, B/1000, b/1000
    return MATERIAL_DENSITY * LENGTH_M * (H_m*b_m + 2*h_m*B_m) * 1000

def calc_strength(H, h, B, b):
    return (4 * YIELD_STRENGTH * calc_I(H, h, B, b)) / (0.0125 * LENGTH_M)

def calc_str_w(H, B, b):
    h = (TOTAL_HEIGHT - H) / 2.0
    return calc_strength(H, h, B, b) / calc_mass(H, h, B, b)

def calc_Mcr(H, h, B, b):
    Iy = calc_Iy(H, h, B, b)
    J = calc_J(H, h, B, b)
    if Iy <= 0 or J <= 0: return 0.0
    return (C1_3PT * np.pi / LENGTH_M) * np.sqrt(E_MODULUS * Iy * G_MODULUS * J)

def calc_Myield(H, h, B, b):
    Ix = calc_I(H, h, B, b)
    y_max = (H/1000 + h/1000)
    if y_max <= 0: return 0.0
    return YIELD_STRENGTH * Ix / y_max

def stability_ratio(H, h, B, b):
    Mcr = calc_Mcr(H, h, B, b)
    My = calc_Myield(H, h, B, b)
    if My <= 0: return 0.0
    return Mcr / My

def find_H_opt(b, B=B_FIXED):
    def obj(H):
        if H < 12.0 or H > 23.4: return 1e10
        h = (TOTAL_HEIGHT - H) / 2.0
        if h < 0 or h > 6.5: return 1e10
        return -calc_str_w(H, B, b)
    return minimize_scalar(obj, bounds=(12.0, 23.4), method='bounded').x

def compute_R_from_params(b, r, dH, B):
    H_phys = find_H_opt(b, B)
    H = np.clip(H_phys + dH, 12.0, 23.4)
    h = (TOTAL_HEIGHT - H) / 2.0
    if h <= 0 or h > 6.5: return 0.0
    return stability_ratio(H, h, B, b)

# ==========================================
# HETEROSCEDASTIC NOISE MODEL
# ==========================================
def get_noise_variance(R):
    sigma_base = 0.001
    sigma_peak = 0.004
    width = 0.5
    log_R = np.log(np.maximum(R, 1e-6))
    bump = np.exp(-0.5 * (log_R / width)**2)
    return sigma_base + sigma_peak * bump

def get_alpha_array(X_5d):
    return np.array([get_noise_variance(x[4]) for x in X_5d])

# ==========================================
# CONSTRAINTS & TRANSFORMS
# ==========================================
def check_3d(b, r, delta_H):
    H_phys = find_H_opt(b)
    H = H_phys + delta_H
    h = (TOTAL_HEIGHT - H) / 2.0
    if not (12.0 <= H <= 23.4 and 0 <= h <= 6.5): return False
    if not (MIN_WEB_THICKNESS <= b <= MAX_WEB_RATIO*B_FIXED): return False
    return 0 <= r <= (B_FIXED - b)/2.0

def enforce_3d(params):
    b, r, delta_H = params
    b = np.clip(b, BOUNDS_5D['b'][0], min(MAX_WEB_RATIO*B_FIXED, BOUNDS_5D['b'][1]))
    r = np.clip(r, 0, min((B_FIXED-b)/2.0, BOUNDS_5D['r'][1]))
    H_phys = find_H_opt(b)
    delta_H = np.clip(delta_H, BOUNDS_5D['delta_H'][0], BOUNDS_5D['delta_H'][1])
    H = np.clip(H_phys + delta_H, 12.0, 23.4)
    delta_H = H - H_phys
    h = (TOTAL_HEIGHT - H) / 2.0
    return np.array([b, r, delta_H]), H, h

def transform_raw_to_5d(X_raw):
    """Transform raw (H, B, b, r) to (b, r, dH, B, R)"""
    X_5d = []
    for i in range(len(X_raw)):
        H, B, b, r = X_raw[i]
        H_phys = find_H_opt(b, B)
        dH = H - H_phys
        h = (TOTAL_HEIGHT - H) / 2.0
        R = stability_ratio(H, h, B, b) if h > 0 else 0.0
        X_5d.append([b, r, dH, B, R])
    return np.array(X_5d)

def x5d_from_3d(b, r, dH, B=B_FIXED):
    R = compute_R_from_params(b, r, dH, B)
    return np.array([b, r, dH, B, R])

def normalize(X):
    X_n = np.copy(X).astype(float)
    for i, k in enumerate(PARAM_KEYS):
        rng = BOUNDS_5D[k][1] - BOUNDS_5D[k][0]
        X_n[:, i] = (X[:, i] - BOUNDS_5D[k][0]) / rng if rng > 0 else 0.0
    return X_n

def denormalize(X_n):
    X = np.copy(X_n)
    for i, k in enumerate(PARAM_KEYS):
        X[:, i] = X_n[:, i] * (BOUNDS_5D[k][1] - BOUNDS_5D[k][0]) + BOUNDS_5D[k][0]
    return X

# ==========================================
# GP TRAINING
# ==========================================
def train_gp(X_5d, y):
    X_norm = normalize(X_5d)
    y_log = np.log(y)
    y_mean, y_cent = np.mean(y_log), y_log - np.mean(y_log)
    alpha_array = get_alpha_array(X_5d)

    kernel = ConstantKernel(1.0, (0.1, 10.0)) * Matern(
        length_scale=[0.5]*5, length_scale_bounds=(0.1, 3.0), nu=2.5)

    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=25,
                                   alpha=alpha_array, normalize_y=False)
    gp.fit(X_norm, y_cent)
    return gp, X_norm, y_cent, y_mean

# ==========================================
# ACQUISITION FUNCTIONS
# ==========================================
def ei_acq(gp, X_norm, y_best, xi):
    mu, sigma = gp.predict(X_norm, return_std=True)
    sigma = np.maximum(sigma, 1e-9)
    imp = mu - y_best - xi
    Z = imp / sigma
    return imp * norm.cdf(Z) + sigma * norm.pdf(Z), mu, sigma

def ucb_acq(gp, X_norm, kappa):
    mu, sigma = gp.predict(X_norm, return_std=True)
    return mu + kappa * np.maximum(sigma, 1e-9), mu, sigma

def optimize_acq(gp, acq_fn, acq_params, n_cand=4000):
    valid_5d, valid_norm = [], []
    attempts = 0
    while len(valid_5d) < n_cand and attempts < 5000:
        attempts += 1
        b = np.random.uniform(*BOUNDS_5D['b'])
        r = np.random.uniform(0, min((B_FIXED-b)/2.0, BOUNDS_5D['r'][1]))
        dH = np.random.uniform(*BOUNDS_5D['delta_H'])
        if check_3d(b, r, dH):
            x5 = x5d_from_3d(b, r, dH)
            valid_5d.append(x5)
            valid_norm.append(normalize(x5.reshape(1, -1))[0])
    if not valid_5d: raise ValueError("No valid candidates!")
    valid_norm = np.array(valid_norm)
    acq_vals, mu_vals, sig_vals = acq_fn(gp, valid_norm, **acq_params)
    idx = np.argmax(acq_vals)
    return valid_5d[idx][:3], acq_vals[idx], mu_vals[idx], sig_vals[idx]

# ==========================================
# RECOMMENDATION GENERATORS
# ==========================================
def gen_mixed(gp, y_cent, y_mean, X_5d, y_train):
    np.random.seed(100)
    print(f"\n{'='*70}\nMIXED: 2 EI + 2 UCB moderate, then 2 fantasy\n{'='*70}")
    recs, y_best = [], y_cent.max()

    xi = ACQ_CONFIG['EI']['moderate']
    x, _, mu, sig = optimize_acq(gp, ei_acq, {'y_best': y_best, 'xi': xi})
    x, H, h = enforce_3d(x)
    recs.append({'Beam': 1, 'Method': 'EI', 'Param': f'xi={xi:.4f}', 'Type': 'Mod',
                 'H': H, 'B': B_FIXED, 'b': x[0], 'r': x[1], 'h': h, 'dH': x[2],
                 'Pred': np.exp(mu + y_mean), 'Var': sig})

    kappa = ACQ_CONFIG['UCB']['moderate']
    x, _, mu, sig = optimize_acq(gp, ucb_acq, {'kappa': kappa})
    x, H, h = enforce_3d(x)
    recs.append({'Beam': 2, 'Method': 'UCB', 'Param': f'k={kappa:.4f}', 'Type': 'Mod',
                 'H': H, 'B': B_FIXED, 'b': x[0], 'r': x[1], 'h': h, 'dH': x[2],
                 'Pred': np.exp(mu + y_mean), 'Var': sig})

    X_fant = np.vstack([X_5d,
        x5d_from_3d(recs[0]['b'], recs[0]['r'], recs[0]['dH']).reshape(1,-1),
        x5d_from_3d(recs[1]['b'], recs[1]['r'], recs[1]['dH']).reshape(1,-1)])
    y_fant = np.append(y_train, [recs[0]['Pred'], recs[1]['Pred']])
    gp_f, _, y_f_cent, y_f_mean = train_gp(X_fant, y_fant)
    y_best_f = y_f_cent.max()

    for i, (meth, typ) in enumerate([('EI','exploit'),('EI','explore'),
                                      ('UCB','exploit'),('UCB','explore')], 3):
        if meth == 'EI':
            xi = ACQ_CONFIG['EI'][typ]
            x, _, mu, sig = optimize_acq(gp_f, ei_acq, {'y_best': y_best_f, 'xi': xi})
        else:
            kappa = ACQ_CONFIG['UCB'][typ]
            x, _, mu, sig = optimize_acq(gp_f, ucb_acq, {'kappa': kappa})
        x, H, h = enforce_3d(x)
        recs.append({'Beam': i, 'Method': meth, 'Type': typ.title(),
                     'H': H, 'B': B_FIXED, 'b': x[0], 'r': x[1], 'h': h, 'dH': x[2],
                     'Pred': np.exp(mu + y_f_mean), 'Var': sig})
    return pd.DataFrame(recs)

def gen_ucb(gp, y_mean, X_5d, y_train, y_cent_orig):
    np.random.seed(100)
    print(f"\n{'='*70}\nPURE UCB: 3 beams\n{'='*70}")
    recs, kappa = [], ACQ_CONFIG['UCB']['moderate']
    current_gp, current_X, current_y = gp, X_5d.copy(), y_train.copy()
    current_y_mean = y_mean

    for i in range(3):
        x, _, mu, sig = optimize_acq(current_gp, ucb_acq, {'kappa': kappa})
        x, H, h = enforce_3d(x)
        pred = np.exp(mu + current_y_mean)
        recs.append({'Beam': i+1, 'Method': 'UCB', 'Type': 'Mod',
                     'H': H, 'B': B_FIXED, 'b': x[0], 'r': x[1], 'h': h, 'dH': x[2],
                     'Pred': pred, 'Var': sig})
        if i < 2:
            current_X = np.vstack([current_X, x5d_from_3d(x[0], x[1], x[2]).reshape(1,-1)])
            current_y = np.append(current_y, pred)
            current_gp, _, _, current_y_mean = train_gp(current_X, current_y)
    return pd.DataFrame(recs)

def gen_ei(gp, y_cent, y_mean, X_5d, y_train):
    np.random.seed(100)
    print(f"\n{'='*70}\nPURE EI: 3 beams\n{'='*70}")
    recs, xi = [], ACQ_CONFIG['EI']['moderate']
    current_gp, current_X, current_y = gp, X_5d.copy(), y_train.copy()
    current_y_mean, current_y_cent = y_mean, y_cent.copy()

    for i in range(3):
        y_best = current_y_cent.max()
        x, _, mu, sig = optimize_acq(current_gp, ei_acq, {'y_best': y_best, 'xi': xi})
        x, H, h = enforce_3d(x)
        pred = np.exp(mu + current_y_mean)
        recs.append({'Beam': i+1, 'Method': 'EI', 'Type': 'Mod',
                     'H': H, 'B': B_FIXED, 'b': x[0], 'r': x[1], 'h': h, 'dH': x[2],
                     'Pred': pred, 'Var': sig})
        if i < 2:
            current_X = np.vstack([current_X, x5d_from_3d(x[0], x[1], x[2]).reshape(1,-1)])
            current_y = np.append(current_y, pred)
            current_gp, _, current_y_cent, current_y_mean = train_gp(current_X, current_y)
    return pd.DataFrame(recs)

# ==========================================
# PLOTTING HELPERS
# ==========================================
def get_point_sizes(X_norm, slice_point_norm):
    distances = np.linalg.norm(X_norm - slice_point_norm, axis=1)
    sizes = np.zeros_like(distances)
    colors = np.zeros_like(distances)
    for mask, sz, cl in [
        (distances < 0.15, 250, 0.9), ((distances >= 0.15) & (distances < 0.30), 180, 0.7),
        ((distances >= 0.30) & (distances < 0.50), 120, 0.5),
        ((distances >= 0.50) & (distances < 0.75), 70, 0.3), (distances >= 0.75, 30, 0.1)]:
        sizes[mask] = sz; colors[mask] = cl
    return sizes, distances, colors

def make_test_line(slice_pt_norm, dim, n=100):
    """Generate test points sweeping dim, recomputing R for physical dims."""
    X_test = np.tile(slice_pt_norm, (n, 1))
    X_test[:, dim] = np.linspace(0, 1, n)
    if dim != 4:
        X_real = denormalize(X_test)
        for j in range(n):
            X_real[j, 4] = compute_R_from_params(X_real[j,0], X_real[j,1], X_real[j,2], X_real[j,3])
        X_test = normalize(X_real)
    return X_test

# ==========================================
# 1D GLOBAL PLOTS
# ==========================================
def plot_1d_global(gp, X_norm, y, y_mean, rec_list):
    print("\nPlotting 1D global slices (4 dims: b, r, dH, R)...")
    slice_pt = np.mean(X_norm, axis=0)
    slice_real = denormalize(slice_pt.reshape(1, -1))[0]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'I-Beam 5D: Global 1D Slices\n'
                 f'(b={slice_real[0]:.2f}, r={slice_real[1]:.2f}, dH={slice_real[2]:.2f}, '
                 f'B={slice_real[3]:.1f}, R={slice_real[4]:.3f})',
                 fontsize=14, weight='bold')

    for ax_idx, dim in enumerate(PLOT_DIMS):
        ax = axes[ax_idx]
        X_test = make_test_line(slice_pt, dim)
        mu, sig = gp.predict(X_test, return_std=True)
        mu_r = np.exp(mu + y_mean)
        epistemic_std = np.exp(mu + y_mean) * sig
        X_test_real = denormalize(X_test)
        noise_var = np.array([get_noise_variance(x[4]) for x in X_test_real])
        aleatory_std = np.exp(mu + y_mean) * np.sqrt(noise_var)
        total_std = np.sqrt(epistemic_std**2 + aleatory_std**2)
        x_vals = X_test_real[:, dim]

        ax.plot(x_vals, mu_r, 'b-', lw=2, label='Mean prediction')
        ax.fill_between(x_vals, mu_r - 2*total_std, mu_r + 2*total_std,
                        alpha=0.25, color='orange', label='+-2s aleatory')
        ax.fill_between(x_vals, mu_r - 2*epistemic_std, mu_r + 2*epistemic_std,
                        alpha=0.3, color='blue', label='+-2s epistemic')

        X_den = denormalize(X_norm)
        ax.scatter(X_den[:, dim], y, c='red', s=50, alpha=0.6, ec='black', lw=1,
                  label='Data', zorder=5)

        for rec, clr, mrk, lbl in zip(rec_list, ['lime','cyan','yellow'], ['*','s','D'],
                                       ['Mixed','UCB','EI']):
            if rec is not None:
                rec_R = [compute_R_from_params(row['b'], row['r'], row['dH']) for _, row in rec.iterrows()]
                rec_vals = {'b': rec['b'].values, 'r': rec['r'].values, 'dH': rec['dH'].values}
                rec_vals['R'] = np.array(rec_R)
                dim_key = PARAM_KEYS[dim]
                if dim_key in rec_vals:
                    ax.scatter(rec_vals[dim_key], rec['Pred'], c=clr, s=150, marker=mrk,
                              ec='black', lw=1.5, label=lbl, zorder=10)

        if dim == 2: ax.axvline(0, color='green', ls='--', alpha=0.5, lw=2, label='Physics opt')
        if dim == 4: ax.axvline(1.0, color='red', ls='--', alpha=0.5, lw=2, label='R=1 (tip boundary)')

        ax.set_xlabel(PLOT_NAMES[ax_idx], fontsize=11)
        ax.set_ylabel('Str/w (N/g)', fontsize=11)
        ax.set_title(PLOT_NAMES[ax_idx], fontsize=12, weight='bold')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('ibeam5d_1d_global.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# 1D SLICE PLOTS
# ==========================================
def plot_slice_1d(gp, X_norm, y, y_mean, slice_point_norm, slice_name, slice_idx):
    slice_real = denormalize(slice_point_norm.reshape(1, -1))[0]
    point_sizes, point_distances, point_colors = get_point_sizes(X_norm, slice_point_norm)
    on_slice = point_distances < 0.1

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'Slice {slice_idx}: {slice_name}\n'
                 f'(b={slice_real[0]:.2f}, r={slice_real[1]:.2f}, dH={slice_real[2]:.2f}, '
                 f'B={slice_real[3]:.1f}, R={slice_real[4]:.3f})',
                 fontsize=14, weight='bold')

    for ax_idx, dim in enumerate(PLOT_DIMS):
        ax = axes[ax_idx]
        X_test = make_test_line(slice_point_norm, dim)
        mu, sig = gp.predict(X_test, return_std=True)
        mu_r = np.exp(mu + y_mean)
        epistemic_std = np.exp(mu + y_mean) * sig
        X_test_real = denormalize(X_test)
        noise_var = np.array([get_noise_variance(x[4]) for x in X_test_real])
        aleatory_std = np.exp(mu + y_mean) * np.sqrt(noise_var)
        total_std = np.sqrt(epistemic_std**2 + aleatory_std**2)
        x_vals = X_test_real[:, dim]

        ax.plot(x_vals, mu_r, 'b-', lw=2, label='Mean')
        ax.fill_between(x_vals, mu_r - 2*total_std, mu_r + 2*total_std,
                        alpha=0.25, color='orange', label='+-2s aleatory')
        ax.fill_between(x_vals, mu_r - 2*epistemic_std, mu_r + 2*epistemic_std,
                        alpha=0.3, color='blue', label='+-2s epistemic')

        X_den = denormalize(X_norm)
        ax.scatter(X_den[:, dim], y, s=point_sizes, c=point_colors, cmap='Reds',
                  vmin=0, vmax=1, alpha=0.6, ec='black', lw=0.5, zorder=5,
                  label='Data (by distance)')
        if np.any(on_slice):
            ax.scatter(X_den[on_slice, dim], y[on_slice], s=400, marker='*',
                      c='yellow', ec='black', lw=2, zorder=10, label='On slice')
        if dim == 2: ax.axvline(0, color='green', ls='--', alpha=0.5, lw=2)
        if dim == 4: ax.axvline(1.0, color='red', ls='--', alpha=0.5, lw=2, label='R=1')

        ax.set_xlabel(PLOT_NAMES[ax_idx], fontsize=11)
        ax.set_ylabel('Str/w (N/g)', fontsize=11)
        ax.set_title(PLOT_NAMES[ax_idx], fontsize=12, weight='bold')
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    clean = slice_name.replace('#','num').replace('(','').replace(')','').replace('=','').replace(':','').replace('/','_').replace(' ','_')
    plt.savefig(f'ibeam5d_slice_{slice_idx:02d}_{clean}_1d.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# 2D GLOBAL PLOTS
# ==========================================
def plot_2d_global(gp, X_norm, y, y_mean, rec_list):
    print("\nPlotting 2D global contours (6 combos of b, r, dH, R)...")
    slice_pt = np.mean(X_norm, axis=0)
    slice_real = denormalize(slice_pt.reshape(1, -1))[0]

    from itertools import combinations
    combos = list(combinations(range(len(PLOT_DIMS)), 2))

    for c1, c2 in combos:
        d1, d2 = PLOT_DIMS[c1], PLOT_DIMS[c2]
        n1, n2 = PLOT_NAMES[c1], PLOT_NAMES[c2]

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'5D GP: {n1} vs {n2}\n(other dims at mean)', fontsize=14, weight='bold')

        res = 50
        grid_x, grid_y = np.meshgrid(np.linspace(0, 1, res), np.linspace(0, 1, res))
        X_test = np.tile(slice_pt, (res*res, 1))
        X_test[:, d1] = grid_x.ravel()
        X_test[:, d2] = grid_y.ravel()

        if d1 != 4 and d2 != 4:
            X_real = denormalize(X_test)
            for j in range(len(X_real)):
                X_real[j, 4] = compute_R_from_params(X_real[j,0], X_real[j,1], X_real[j,2], X_real[j,3])
            X_test = normalize(X_real)

        mu, sig = gp.predict(X_test, return_std=True)
        mu_r = np.exp(mu + y_mean).reshape(res, res)
        sig_r = sig.reshape(res, res)

        x_v = np.linspace(BOUNDS_5D[PARAM_KEYS[d1]][0], BOUNDS_5D[PARAM_KEYS[d1]][1], res)
        y_v = np.linspace(BOUNDS_5D[PARAM_KEYS[d2]][0], BOUNDS_5D[PARAM_KEYS[d2]][1], res)
        ext = [x_v[0], x_v[-1], y_v[0], y_v[-1]]

        im1 = axes[0].imshow(mu_r, origin='lower', extent=ext, aspect='auto', cmap='viridis')
        axes[0].contour(mu_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
        axes[0].set_xlabel(n1, fontsize=12); axes[0].set_ylabel(n2, fontsize=12)
        axes[0].set_title('Mean Prediction', fontsize=13, weight='bold')
        X_den = denormalize(X_norm)
        axes[0].scatter(X_den[:, d1], X_den[:, d2], c='red', s=80, ec='white', lw=2, zorder=10)
        if d2 == 2: axes[0].axhline(0, color='green', ls='--', alpha=0.7, lw=2)
        if d1 == 2: axes[0].axvline(0, color='green', ls='--', alpha=0.7, lw=2)
        if d2 == 4: axes[0].axhline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        if d1 == 4: axes[0].axvline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        plt.colorbar(im1, ax=axes[0], label='Str/w (N/g)')

        im2 = axes[1].imshow(sig_r, origin='lower', extent=ext, aspect='auto', cmap='hot')
        axes[1].contour(sig_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
        axes[1].set_xlabel(n1, fontsize=12); axes[1].set_ylabel(n2, fontsize=12)
        axes[1].set_title('Uncertainty (sigma)', fontsize=13, weight='bold')
        axes[1].scatter(X_den[:, d1], X_den[:, d2], c='cyan', s=80, ec='white', lw=2, zorder=10)
        if d2 == 2: axes[1].axhline(0, color='green', ls='--', alpha=0.7, lw=2)
        if d1 == 2: axes[1].axvline(0, color='green', ls='--', alpha=0.7, lw=2)
        if d2 == 4: axes[1].axhline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        if d1 == 4: axes[1].axvline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        plt.colorbar(im2, ax=axes[1], label='Std Dev')

        plt.tight_layout()
        plt.savefig(f'ibeam5d_2d_global_{PARAM_KEYS[d1]}_{PARAM_KEYS[d2]}.png', dpi=150, bbox_inches='tight')
        plt.show()

# ==========================================
# 2D SLICE PLOTS
# ==========================================
def plot_slice_2d(gp, X_norm, y, y_mean, slice_point_norm, slice_name, slice_idx):
    slice_real = denormalize(slice_point_norm.reshape(1, -1))[0]
    point_sizes, point_distances, point_colors = get_point_sizes(X_norm, slice_point_norm)
    on_slice = point_distances < 0.1

    from itertools import combinations
    combos = list(combinations(range(len(PLOT_DIMS)), 2))

    for c1, c2 in combos:
        d1, d2 = PLOT_DIMS[c1], PLOT_DIMS[c2]
        n1, n2 = PLOT_NAMES[c1], PLOT_NAMES[c2]

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'Slice {slice_idx}: {slice_name}\n{n1} vs {n2}', fontsize=14, weight='bold')

        res = 50
        grid_x, grid_y = np.meshgrid(np.linspace(0, 1, res), np.linspace(0, 1, res))
        X_test = np.tile(slice_point_norm, (res*res, 1))
        X_test[:, d1] = grid_x.ravel()
        X_test[:, d2] = grid_y.ravel()

        if d1 != 4 and d2 != 4:
            X_real = denormalize(X_test)
            for j in range(len(X_real)):
                X_real[j, 4] = compute_R_from_params(X_real[j,0], X_real[j,1], X_real[j,2], X_real[j,3])
            X_test = normalize(X_real)

        mu, sig = gp.predict(X_test, return_std=True)
        mu_r = np.exp(mu + y_mean).reshape(res, res)
        sig_r = sig.reshape(res, res)

        x_v = np.linspace(BOUNDS_5D[PARAM_KEYS[d1]][0], BOUNDS_5D[PARAM_KEYS[d1]][1], res)
        y_v = np.linspace(BOUNDS_5D[PARAM_KEYS[d2]][0], BOUNDS_5D[PARAM_KEYS[d2]][1], res)
        ext = [x_v[0], x_v[-1], y_v[0], y_v[-1]]

        X_den = denormalize(X_norm)

        im1 = axes[0].imshow(mu_r, origin='lower', extent=ext, aspect='auto', cmap='viridis')
        axes[0].contour(mu_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
        axes[0].scatter(X_den[:, d1], X_den[:, d2], s=point_sizes, c=point_colors,
                       cmap='Reds', vmin=0, vmax=1, alpha=0.6, ec='white', lw=1, zorder=5)
        if np.any(on_slice):
            axes[0].scatter(X_den[on_slice, d1], X_den[on_slice, d2], s=400, marker='*',
                           c='yellow', ec='black', lw=2, zorder=10)
        axes[0].set_xlabel(n1); axes[0].set_ylabel(n2)
        axes[0].set_title('Mean', weight='bold')
        if d2 == 4: axes[0].axhline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        if d1 == 4: axes[0].axvline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        plt.colorbar(im1, ax=axes[0], label='Str/w')

        im2 = axes[1].imshow(sig_r, origin='lower', extent=ext, aspect='auto', cmap='hot')
        axes[1].contour(sig_r, origin='lower', extent=ext, colors='white', alpha=0.4, levels=10)
        axes[1].scatter(X_den[:, d1], X_den[:, d2], s=point_sizes, c=point_colors,
                       cmap='Blues', vmin=0, vmax=1, alpha=0.6, ec='white', lw=1, zorder=5)
        if np.any(on_slice):
            axes[1].scatter(X_den[on_slice, d1], X_den[on_slice, d2], s=400, marker='*',
                           c='yellow', ec='black', lw=2, zorder=10)
        axes[1].set_xlabel(n1); axes[1].set_ylabel(n2)
        axes[1].set_title('Uncertainty', weight='bold')
        if d2 == 4: axes[1].axhline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        if d1 == 4: axes[1].axvline(1.0, color='red', ls='--', alpha=0.7, lw=2)
        plt.colorbar(im2, ax=axes[1], label='Std Dev')

        plt.tight_layout()
        clean = slice_name.replace('#','num').replace('(','').replace(')','').replace('=','').replace(':','').replace('/','_').replace(' ','_')
        plt.savefig(f'ibeam5d_slice_{slice_idx:02d}_{clean}_{PARAM_KEYS[d1]}_{PARAM_KEYS[d2]}.png', dpi=150, bbox_inches='tight')
        plt.show()

# ==========================================
# DATA LOADING
# ==========================================
def load_data():
    import urllib.request, io
    DATA_URL = "https://raw.githubusercontent.com/andrewvoss8-boop/core-me-data-science-activities-public/main/data/I_beam_data.csv"
    print("Downloading data from GitHub...")
    with urllib.request.urlopen(DATA_URL) as response:
        csv_data = response.read().decode('utf-8')
    df = pd.read_csv(io.StringIO(csv_data))
    param_names = ['H_web_height', 'B_flange_width', 'b_web_thick', 'r_fillet']
    df_clean = df[param_names + ['Str/w (N/g)']].dropna()

    def meets_constraints(row):
        H, B, b, r = row['H_web_height'], row['B_flange_width'], row['b_web_thick'], row['r_fillet']
        h = (TOTAL_HEIGHT - H) / 2.0
        if B < MIN_FLANGE_WIDTH or b < MIN_WEB_THICKNESS or b > MAX_WEB_RATIO * B: return False
        if r < 0 or r > (B - b) / 2.0: return False
        if h < 0 or h > 6.5: return False
        return True

    df_ok = df_clean[df_clean.apply(meets_constraints, axis=1)].copy()
    print(f"Loaded {len(df_ok)} valid beams, Str/w: [{df_ok['Str/w (N/g)'].min():.2f}, {df_ok['Str/w (N/g)'].max():.2f}]")
    return df_ok

# ==========================================
# MAIN
# ==========================================
if __name__ == "__main__":
    print("="*70)
    print("I-BEAM 5D OPTIMIZER WITH STABILITY RATIO")
    print("="*70)

    df = load_data()
    X_raw = df[['H_web_height', 'B_flange_width', 'b_web_thick', 'r_fillet']].values
    y = df['Str/w (N/g)'].values
    X_5d = transform_raw_to_5d(X_raw)

    # Calibrate R bounds from data
    R_vals = X_5d[:, 4]
    BOUNDS_5D['R'] = (0.0, max(6.0, np.ceil(R_vals.max())))
    print(f"R range in data: [{R_vals.min():.3f}, {R_vals.max():.3f}]")
    print(f"R bounds set to: {BOUNDS_5D['R']}")

    X_norm = normalize(X_5d)

    print(f"\n5D data (b, r, dH, B, R):")
    print(f"{'#':>3} {'b':>5} {'r':>5} {'dH':>6} {'B':>5} {'R':>6} {'Str/w':>6} {'noise':>8}")
    for i in range(len(y)):
        nv = get_noise_variance(X_5d[i, 4])
        print(f"{i+1:3d} {X_5d[i,0]:5.2f} {X_5d[i,1]:5.2f} {X_5d[i,2]:6.2f} "
              f"{X_5d[i,3]:5.1f} {X_5d[i,4]:6.3f} {y[i]:6.2f} {nv:8.5f}")

    print("\nTraining 5D GP...")
    gp, X_norm, y_cent, y_mean = train_gp(X_5d, y)
    scales = gp.kernel_.k2.length_scale
    print(f"ARD scales: b={scales[0]:.3f}, r={scales[1]:.3f}, dH={scales[2]:.3f}, "
          f"B={scales[3]:.3f}, R={scales[4]:.3f}")

    best_idx = np.argmax(y)
    print(f"\nBest beam #{best_idx}: b={X_5d[best_idx,0]:.2f}, r={X_5d[best_idx,1]:.2f}, "
          f"dH={X_5d[best_idx,2]:.2f}, B={X_5d[best_idx,3]:.1f}, R={X_5d[best_idx,4]:.3f} "
          f"-> Str/w={y[best_idx]:.2f}")

    rec_mixed = gen_mixed(gp, y_cent, y_mean, X_5d, y) if GENERATE_MIXED else None
    rec_ucb = gen_ucb(gp, y_mean, X_5d, y, y_cent) if GENERATE_UCB else None
    rec_ei = gen_ei(gp, y_cent, y_mean, X_5d, y) if GENERATE_EI else None

    for name, rec in [('MIXED', rec_mixed), ('UCB', rec_ucb), ('EI', rec_ei)]:
        if rec is not None:
            print(f"\n{'='*70}\n{name} RECOMMENDATIONS\n{'='*70}")
            print(rec.to_string(index=False))

    rec_list = [rec_mixed, rec_ucb, rec_ei]

    print("\n" + "="*70)
    print("GENERATING PLOTS")
    print("="*70)

    plot_1d_global(gp, X_norm, y, y_mean, rec_list)
    plot_2d_global(gp, X_norm, y, y_mean, rec_list)

    print("\n" + "="*70)
    print("SLICE PLOTS")
    print("="*70)

    slices = []
    top3 = np.argsort(y)[-3:][::-1]
    for i, idx in enumerate(top3):
        slices.append((X_5d[idx], f"Top #{i+1} (Str/w={y[idx]:.2f})"))

    for name, rec in [('Mixed', rec_mixed), ('UCB', rec_ucb), ('EI', rec_ei)]:
        if rec is not None:
            for _, row in rec.iterrows():
                x5 = x5d_from_3d(row['b'], row['r'], row['dH'])
                slices.append((x5, f"{name} Beam {row['Beam']}"))

    for idx, (pt, name) in enumerate(slices):
        pt_norm = normalize(pt.reshape(1, -1))[0]
        print(f"\nSlice {idx+1}/{len(slices)}: {name}")
        plot_slice_1d(gp, X_norm, y, y_mean, pt_norm, name, idx+1)
        plot_slice_2d(gp, X_norm, y, y_mean, pt_norm, name, idx+1)

    print("\n" + "="*70)
    print("DONE!")
    print("="*70)
